In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [17]:
# build the vocabulary of characters and mappings to/from integers

words = open('../data/names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(f"{vocab_size=}")

# build the dataset
block_size = 10 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context) # its just appending the list objects, not the list values
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(f"{X.shape, Y.shape}=")
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

vocab_size=27
(torch.Size([182625, 10]), torch.Size([182625]))=
(torch.Size([22655, 10]), torch.Size([22655]))=
(torch.Size([22866, 10]), torch.Size([22866]))=


In [18]:
torch.manual_seed(42)
n_embd = 10
n_hidden = 200

class FFNN(nn.Module):
  
  def __init__(self, vocab_size, block_size, n_embd, n_hidden):
    super().__init__()
    
    self.emb = nn.Embedding(vocab_size, n_embd)
    
    self.fc1 = nn.Linear(block_size * n_embd, n_hidden,bias=False)
    self.bn1 = nn.BatchNorm1d(n_hidden)
    self.t1 = nn.Tanh()
    
    
    self.fc2 = nn.Linear(n_hidden, n_hidden,bias=False)
    self.bn2 = nn.BatchNorm1d(n_hidden)
    self.t2 = nn.Tanh()


    self.fc3 = nn.Linear(n_hidden, n_hidden,bias=False)
    self.bn3 = nn.BatchNorm1d(n_hidden)
    self.t3 = nn.Tanh()


    self.fc4 = nn.Linear(n_hidden, n_hidden,bias=False)
    self.bn4 = nn.BatchNorm1d(n_hidden)
    self.t4 = nn.Tanh()


    self.fc5 = nn.Linear(n_hidden, vocab_size,bias=False)
    self.bn5 = nn.BatchNorm1d(vocab_size)
    
    
  def forward(self, x):
    x = self.emb(x) # (B,T,C)
    x = x.view(x.shape[0], -1) # (B, T*C)
    
    x1 = self.fc1(x)
    x1 = self.bn1(x1)
    x1 = self.t1(x1)

    x2 = self.fc2(x1)
    x2 = self.bn2(x2)
    x2 = self.t2(x2)

    x3 = self.fc3(x2)
    x3 = self.bn3(x3)
    x3 = self.t3(x3)

    x4 = self.fc4(x3)
    x4 = self.bn4(x4)
    x4 = self.t4(x4)

    logits = self.fc5(x4)
    logits = self.bn5(logits)

    return logits
  
model = FFNN(vocab_size, block_size, n_embd, n_hidden)
print(model)

# model.parameters() is a generator for the layers of the model where parameters are 2d learnable tensors (weights and biases)
print(f"total number of parameters: {sum(p.numel() for p in model.parameters())}")


  
      
    

FFNN(
  (emb): Embedding(27, 10)
  (fc1): Linear(in_features=100, out_features=200, bias=False)
  (bn1): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (t1): Tanh()
  (fc2): Linear(in_features=200, out_features=200, bias=False)
  (bn2): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (t2): Tanh()
  (fc3): Linear(in_features=200, out_features=200, bias=False)
  (bn3): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (t3): Tanh()
  (fc4): Linear(in_features=200, out_features=200, bias=False)
  (bn4): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (t4): Tanh()
  (fc5): Linear(in_features=200, out_features=27, bias=False)
  (bn5): BatchNorm1d(27, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
total number of parameters: 147324


In [20]:
out = model(Xtr[:10])
F.cross_entropy(out, Ytr[:10])

tensor(3.3091, grad_fn=<NllLossBackward0>)

In [24]:
# same optimization as last time
torch.manual_seed(42)

model = FFNN(vocab_size, block_size, n_embd, n_hidden)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

max_steps = 2000
batch_size = 128
lossi = []

for epoch in range(max_steps):
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  x = model(Xb)
  loss = criterion(x, Yb)
  
  # backward pass
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if (epoch + 1) % 100 == 0:
          print(f"Epoch [{epoch+1}/{max_steps}], Loss: {loss.item():.4f}")
 

Epoch [100/2000], Loss: 2.5353
Epoch [200/2000], Loss: 2.3550
Epoch [300/2000], Loss: 2.3911
Epoch [400/2000], Loss: 2.2791
Epoch [500/2000], Loss: 2.2265
Epoch [600/2000], Loss: 2.1836
Epoch [700/2000], Loss: 2.0172
Epoch [800/2000], Loss: 2.2184
Epoch [900/2000], Loss: 2.3035
Epoch [1000/2000], Loss: 2.0101
Epoch [1100/2000], Loss: 2.1209
Epoch [1200/2000], Loss: 2.2881
Epoch [1300/2000], Loss: 2.0976
Epoch [1400/2000], Loss: 2.1615
Epoch [1500/2000], Loss: 2.0722
Epoch [1600/2000], Loss: 2.1716
Epoch [1700/2000], Loss: 2.1235
Epoch [1800/2000], Loss: 2.2094
Epoch [1900/2000], Loss: 2.0885
Epoch [2000/2000], Loss: 2.2673


In [ ]:
torch.manual_seed(42)

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

C = nn.Embedding(vocab_size, n_embd) # embedding layer
layers = [
  nn.Linear(n_embd * block_size, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, n_hidden, bias=False), nn.BatchNorm1d(n_hidden), nn.Tanh(),
  nn.Linear(           n_hidden, vocab_size, bias=False), nn.BatchNorm1d(vocab_size),
]

with torch.no_grad():
  layers[-1].weight *= 0.1

  for layer in layers[:-1]:
    if isinstance(layer, nn.Linear):
      layer.weight *= 5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.weight.shape[1] for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True